# Диаризация спикеров для parquet с `word_timestamps_json`

Ноутбук:

1. читает parquet-файл;
2. прогоняет каждый аудиофайл через `pyannote/speaker-diarization-community-1`;
3. берёт **exclusive speaker diarization** (если доступна);
4. присваивает спикера каждому слову из `word_timestamps_json`;
5. собирает подряд идущие слова одного спикера в куски;
6. сохраняет результат обратно в parquet.

Основной новый столбец: **`speaker_chunks_json`**.

Дополнительно сохраняются:

- `speaker_diarization_turns_json` — сырые интервалы диаризации;
- `diarization_error` — ошибка обработки строки, если она возникла.

Ниже основной alignment делается по словам, потому что `word_timestamps_json` заполнен стабильнее,
а это даёт более точное сопоставление текста со спикером.


## Что нужно перед запуском

1. Принять условия доступа к модели на Hugging Face.
2. Создать HF token.
3. Убедиться, что аудиофайлы доступны по путям из `absolute_path` **или** настроить `WINDOWS_PATH_REPLACEMENTS`.
4. При необходимости включить GPU.

Пример установки зависимостей:

```bash
pip install -U pandas pyarrow tqdm "pyannote.audio>=4,<5"
```

Если запускаете на CUDA, должны быть совместимые `torch`/`torchaudio`.


In [ ]:
# При необходимости раскомментируйте и выполните один раз.
# %pip install -U pandas pyarrow tqdm "pyannote.audio>=4,<5"


In [1]:
from __future__ import annotations

import ast
import json
import math
import os
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

import torch
from pyannote.audio import Pipeline


In [ ]:
# =========================
# Конфигурация
# =========================

INPUT_PARQUET = Path("recognized_calls_gigaam_v3_e2e_ctc_github.parquet")
OUTPUT_PARQUET = Path("recognized_calls_gigaam_v3_e2e_ctc_github_with_speakers.parquet")

MODEL_ID = "pyannote/speaker-diarization-community-1"
HF_TOKEN = os.environ.get("HF_TOKEN")

USE_GPU = True
NUM_SPEAKERS = None      # например: 2
MIN_SPEAKERS = None      # например: 1
MAX_SPEAKERS = None      # например: 3

# Если parquet хранит Windows-пути, а ноутбук запускается на Linux/macOS,
# настройте замену префиксов.
WINDOWS_PATH_REPLACEMENTS = {
    r"Z:\calls2": "/data/calls2",
}

# Если один и тот же спикер замолкает надолго, удобнее начинать новый кусок.
MAX_PAUSE_BETWEEN_WORDS_SEC = 1.0

In [5]:
# Быстрая проверка входного файла

df = pd.read_parquet(INPUT_PARQUET)
print(f"shape={df.shape}")
print(df.columns.tolist())

df[["absolute_path", "name_file", "word_timestamps_json", "segment_timestamps_json"]].head(2)


shape=(1000, 17)
['call_id', 'transcription', 'fraud', 'created_at', 'id', 'presence_of_blocks', 'block_analysis_result', 'call_type', 'product_type', 'deep_analytic_enqueued', 'absolute_path', 'name_file', 'predicted_transcription', 'word_timestamps_json', 'segment_timestamps_json', 'status', 'error_message']


,absolute_path,name_file,word_timestamps_json,segment_timestamps_json
0,Z:\calls2\1773820842.1776908.wav,1773820842.1776908.wav,"[{""text"": ""Алло."", ""start"": 0.031, ""end"": 3.35...","[{""text"": ""Алло. Алло, добрый день! С Да. Да, ..."
1,Z:\calls2\1773820804.1776814.wav,1773820804.1776814.wav,"[{""text"": ""Ало,ран,"", ""start"": 0.0, ""end"": 2.0...",None


In [6]:
def is_missing(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and value.strip().lower() in {"", "none", "null", "nan"}:
        return True
    return False


def parse_json_list(value: Any, field_name: str) -> list[dict[str, Any]]:
    if is_missing(value):
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, str):
        raw_value = value.strip()
        if raw_value.lower() in {"", "none", "null", "nan"}:
            return []

        try:
            parsed = json.loads(raw_value)
        except json.JSONDecodeError as json_exc:
            try:
                parsed = ast.literal_eval(raw_value)
            except Exception as literal_exc:
                preview = raw_value[:300].replace("\n", "\\n")
                raise ValueError(
                    f"Не удалось распарсить {field_name}. "
                    f"json.loads: {json_exc}. "
                    f"ast.literal_eval: {literal_exc}. "
                    f"Первые символы: {preview}"
                ) from literal_exc

        if not isinstance(parsed, list):
            raise TypeError(f"{field_name} должен быть списком, получено: {type(parsed)}")

        return parsed

    raise TypeError(f"Неожиданный тип для {field_name}: {type(value)}")


def normalize_windows_prefix(path_str: str) -> str:
    return path_str.replace("/", "\\").rstrip("\\").lower()


def apply_path_replacements(path_str: str, replacements: dict[str, str]) -> list[str]:
    candidates = [path_str]
    normalized_path = normalize_windows_prefix(path_str)

    for src_prefix, dst_prefix in replacements.items():
        normalized_src = normalize_windows_prefix(src_prefix)
        if normalized_path.startswith(normalized_src):
            suffix = path_str[len(src_prefix):].lstrip("\\/")
            unix_suffix = suffix.replace("\\", "/")
            candidates.append(str(Path(dst_prefix) / unix_suffix))

    # Дополнительная эвристика: просто заменить backslash на slash.
    if "\\" in path_str:
        candidates.append(path_str.replace("\\", "/"))

    # Убираем дубликаты, сохраняя порядок.
    unique_candidates = []
    seen = set()
    for candidate in candidates:
        if candidate not in seen:
            unique_candidates.append(candidate)
            seen.add(candidate)
    return unique_candidates


def resolve_audio_path(path_str: str, replacements: dict[str, str]) -> Path:
    if is_missing(path_str):
        raise ValueError("Пустой путь до аудио")

    candidates = apply_path_replacements(str(path_str), replacements)
    for candidate in candidates:
        candidate_path = Path(candidate)
        if candidate_path.exists():
            return candidate_path

    joined = "\n - ".join(candidates)
    raise FileNotFoundError(f"Аудиофайл не найден. Проверенные варианты:\n - {joined}")


def overlap_duration(start_a: float, end_a: float, start_b: float, end_b: float) -> float:
    return max(0.0, min(end_a, end_b) - max(start_a, start_b))


def nearest_distance_to_interval(point: float, start: float, end: float) -> float:
    if start <= point <= end:
        return 0.0
    return min(abs(point - start), abs(point - end))


In [7]:
def load_pipeline(model_id: str, hf_token: str, use_gpu: bool = True) -> Pipeline:
    if not hf_token or hf_token == "PASTE_YOUR_HF_TOKEN_HERE":
        raise ValueError(
            "Нужно передать Hugging Face token. Установите переменную окружения HF_TOKEN "
            "или вставьте токен в переменную HF_TOKEN в конфиге."
        )

    pipeline = Pipeline.from_pretrained(model_id, token=hf_token)

    if use_gpu and torch.cuda.is_available():
        pipeline.to(torch.device("cuda"))
        print("Pipeline переведён на CUDA")
    else:
        print("Pipeline работает на CPU")

    return pipeline


def get_best_diarization_annotation(output: Any):
    exclusive = getattr(output, "exclusive_speaker_diarization", None)
    if exclusive is not None:
        return exclusive

    speaker_diarization = getattr(output, "speaker_diarization", None)
    if speaker_diarization is not None:
        return speaker_diarization

    return output


def annotation_to_turns(annotation: Any) -> list[dict[str, Any]]:
    turns = []
    for segment, _, speaker in annotation.itertracks(yield_label=True):
        turns.append(
            {
                "speaker": str(speaker),
                "start": round(float(segment.start), 3),
                "end": round(float(segment.end), 3),
            }
        )

    turns.sort(key=lambda item: (item["start"], item["end"], item["speaker"]))
    return turns


def choose_speaker_for_interval(
    start: float,
    end: float,
    diarization_turns: list[dict[str, Any]],
) -> str:
    if not diarization_turns:
        return "UNKNOWN"

    if end <= start:
        end = start + 1e-3

    candidates = []
    for turn in diarization_turns:
        overlap = overlap_duration(start, end, turn["start"], turn["end"])
        if overlap > 0:
            turn_duration = turn["end"] - turn["start"]
            candidates.append((overlap, turn_duration, turn["speaker"]))

    if candidates:
        candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
        return candidates[0][2]

    midpoint = (start + end) / 2.0
    nearest_turn = min(
        diarization_turns,
        key=lambda turn: nearest_distance_to_interval(midpoint, turn["start"], turn["end"]),
    )
    return nearest_turn["speaker"]


In [8]:
def load_words_from_row(row: pd.Series) -> list[dict[str, Any]]:
    raw_words = parse_json_list(row["word_timestamps_json"], "word_timestamps_json")
    words = []

    for word_index, raw_word in enumerate(raw_words):
        text = str(raw_word.get("text", "")).strip()
        if not text:
            continue

        start = float(raw_word["start"])
        end = float(raw_word["end"])
        words.append(
            {
                "word_index": word_index,
                "text": text,
                "start": round(start, 3),
                "end": round(end, 3),
            }
        )

    return words


def assign_speakers_to_words(
    words: list[dict[str, Any]],
    diarization_turns: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    aligned_words = []
    for word in words:
        speaker = choose_speaker_for_interval(word["start"], word["end"], diarization_turns)
        aligned_words.append({**word, "speaker": speaker})
    return aligned_words


def build_speaker_chunks(
    aligned_words: list[dict[str, Any]],
    max_pause_between_words_sec: float = 1.0,
) -> list[dict[str, Any]]:
    if not aligned_words:
        return []

    chunks: list[dict[str, Any]] = []
    current_words: list[dict[str, Any]] = []

    def flush_current() -> None:
        if not current_words:
            return
        chunks.append(
            {
                "speaker": current_words[0]["speaker"],
                "start": current_words[0]["start"],
                "end": current_words[-1]["end"],
                "text": " ".join(word["text"] for word in current_words),
                "words": current_words.copy(),
            }
        )

    for word in aligned_words:
        if not current_words:
            current_words.append(word)
            continue

        previous_word = current_words[-1]
        same_speaker = word["speaker"] == previous_word["speaker"]
        short_pause = (word["start"] - previous_word["end"]) <= max_pause_between_words_sec

        if same_speaker and short_pause:
            current_words.append(word)
            continue

        flush_current()
        current_words = [word]

    flush_current()
    return chunks


def load_segments_from_row(row: pd.Series) -> list[dict[str, Any]]:
    raw_segments = parse_json_list(row["segment_timestamps_json"], "segment_timestamps_json")
    segments = []

    for segment_index, raw_segment in enumerate(raw_segments):
        segments.append(
            {
                "segment_index": segment_index,
                "text": str(raw_segment.get("text", "")).strip(),
                "start": round(float(raw_segment["start"]), 3),
                "end": round(float(raw_segment["end"]), 3),
            }
        )

    return segments


def align_segments_to_speakers(
    segments: list[dict[str, Any]],
    aligned_words: list[dict[str, Any]],
    diarization_turns: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    results = []

    for segment in segments:
        segment_words = [
            word
            for word in aligned_words
            if overlap_duration(segment["start"], segment["end"], word["start"], word["end"]) > 0
        ]

        if segment_words:
            durations_by_speaker: dict[str, float] = {}
            for word in segment_words:
                durations_by_speaker.setdefault(word["speaker"], 0.0)
                durations_by_speaker[word["speaker"]] += max(word["end"] - word["start"], 1e-3)
            speaker = max(durations_by_speaker, key=durations_by_speaker.get)
        else:
            speaker = choose_speaker_for_interval(segment["start"], segment["end"], diarization_turns)

        results.append({**segment, "speaker": speaker})

    return results


In [ ]:
def diarize_single_row(
    row: pd.Series,
    pipeline: Pipeline,
    path_replacements: dict[str, str],
    max_pause_between_words_sec: float = 1.0,
    build_segment_alignment: bool = True,
    num_speakers: int | None = None,
    min_speakers: int | None = None,
    max_speakers: int | None = None,
) -> dict[str, Any]:
    audio_path = resolve_audio_path(row["absolute_path"], path_replacements)

    diarization_kwargs: dict[str, Any] = {}
    if num_speakers is not None:
        diarization_kwargs["num_speakers"] = num_speakers
    if min_speakers is not None:
        diarization_kwargs["min_speakers"] = min_speakers
    if max_speakers is not None:
        diarization_kwargs["max_speakers"] = max_speakers

    output = pipeline(str(audio_path), **diarization_kwargs)
    annotation = get_best_diarization_annotation(output)
    diarization_turns = annotation_to_turns(annotation)

    words = load_words_from_row(row)
    aligned_words = assign_speakers_to_words(words, diarization_turns)
    speaker_chunks = build_speaker_chunks(
        aligned_words,
        max_pause_between_words_sec=max_pause_between_words_sec,
    )

    return {
        "speaker_chunks_json": json.dumps(speaker_chunks, ensure_ascii=False),
        "diarization_error": None,
    }


def safe_diarize_single_row(
    row: pd.Series,
    pipeline: Pipeline,
    path_replacements: dict[str, str],
    max_pause_between_words_sec: float = 1.0,
    build_segment_alignment: bool = True,
    num_speakers: int | None = None,
    min_speakers: int | None = None,
    max_speakers: int | None = None,
) -> dict[str, Any]:
    try:
        return diarize_single_row(
            row=row,
            pipeline=pipeline,
            path_replacements=path_replacements,
            max_pause_between_words_sec=max_pause_between_words_sec,
            build_segment_alignment=build_segment_alignment,
            num_speakers=num_speakers,
            min_speakers=min_speakers,
            max_speakers=max_speakers,
        )
    except Exception as exc:
        return {
            "speaker_chunks_json": None,
            "diarization_error": f"{type(exc).__name__}: {exc}",
        }

In [10]:
pipeline = load_pipeline(
    model_id=MODEL_ID,
    hf_token=HF_TOKEN,
    use_gpu=USE_GPU,
)


[NeMo W 2026-04-14 12:36:02 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
W0414 12:36:02.306000 15844 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


Pipeline переведён на CUDA


In [11]:
results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Running speaker diarization"):
    result = safe_diarize_single_row(
        row=row,
        pipeline=pipeline,
        path_replacements=WINDOWS_PATH_REPLACEMENTS,
        max_pause_between_words_sec=MAX_PAUSE_BETWEEN_WORDS_SEC,
        build_segment_alignment=BUILD_SEGMENT_ALIGNMENT,
        num_speakers=NUM_SPEAKERS,
        min_speakers=MIN_SPEAKERS,
        max_speakers=MAX_SPEAKERS,
    )
    results.append(result)

results_df = pd.DataFrame(results)
result_df = pd.concat([df.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1)

result_df.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Saved: {OUTPUT_PARQUET.resolve()}")
print(result_df[["speaker_chunks_json", "diarization_error"]].head(2))


Running speaker diarization:   0%|          | 0/1000 [00:00<?, ?it/s]

[NeMo W 2026-04-14 12:36:04 nemo_logging:364] c:\Users\corey\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyannote\audio\utils\reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
    It can be re-enabled by calling
       >>> import torch
       >>> torch.backends.cuda.matmul.allow_tf32 = True
       >>> torch.backends.cudnn.allow_tf32 = True
    See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.
    
      warnings.warn(
    


Saved: D:\project\multicriteria-dialog-audit-ml\giga\recognized_calls_gigaam_v3_e2e_ctc_github_with_speakers.parquet
                                 speaker_chunks_json diarization_error
0  [{"speaker": "SPEAKER_00", "start": 0.031, "en...              None
1  [{"speaker": "SPEAKER_00", "start": 0.0, "end"...              None


In [12]:
# Пример: посмотреть первые speaker-chunks в удобном виде

sample_idx = result_df["speaker_chunks_json"].first_valid_index()
if sample_idx is not None:
    sample_chunks = parse_json_list(result_df.loc[sample_idx, "speaker_chunks_json"], "speaker_chunks_json")
    sample_chunks[:3]
else:
    print("Нет строк с успешной диаризацией")


## Формат `speaker_chunks_json`

В каждой строке parquet в новом столбце будет JSON-массив примерно такого вида:

```json
[
  {
    "speaker": "SPEAKER_00",
    "start": 0.031,
    "end": 4.351,
    "text": "Алло. Алло, добрый день!",
    "words": [
      {"word_index": 0, "text": "Алло.", "start": 0.031, "end": 3.351, "speaker": "SPEAKER_00"},
      {"word_index": 1, "text": "Алло,", "start": 3.511, "end": 3.791, "speaker": "SPEAKER_00"}
    ]
  },
  {
    "speaker": "SPEAKER_01",
    "start": 5.591,
    "end": 6.151,
    "text": "Да. Да,",
    "words": [
      {"word_index": 5, "text": "Да.", "start": 5.591, "end": 5.871, "speaker": "SPEAKER_01"},
      {"word_index": 6, "text": "Да,", "start": 5.991, "end": 6.151, "speaker": "SPEAKER_01"}
    ]
  }
]
```

Это уже удобно использовать дальше для аналитики, визуализации, чанкинга диалогов и speaker-aware post-processing.
